# 믹서 MCP에서 도구 목록을 받아 봐요

이 노트북은 강사가 만든 **실제 MCP 서버 3.3.0**을 Colab에서 실행해요. AI 모델과 실제 믹서는 사용하지 않아요.

1. 첫 번째 코드 칸에서 필요한 프로그램과 **서버 파일까지 내려받아요**.
2. 두 번째 코드 칸에서 도구 24개의 목록을 받아요.
3. 세 번째 코드 칸에서 잘못된 채널 `40`을 보내고, 거부 이유를 확인해요.

순서대로 각 코드 칸 왼쪽의 ▶를 한 번씩 누르세요. 프로그램을 내려받는 칸은 첫 번째 칸이에요. Colab 화면을 쓰는 동안에도 인터넷 연결은 필요해요. 첫 칸은 내려받는 동안 ▶ 자리에 표시가 돌아요. 기다리는 것이 정상이고, 끝나면 `준비됐어요`가 찍혀요. 두 번째와 세 번째 칸은 내려받은 서버 파일을 실행해요.

3단계의 채널40 거부 메시지는 예상한 결과예요. 그 밖에 설치·실행 오류가 나오면 빨간 메시지의 **마지막 줄부터 위로 몇 줄**을 그대로 복사해 강사에게 보여 주세요. 결과가 없는데 성공한 것으로 적지 않아요.

장비 주소를 입력하거나 `connection_connect`를 호출하지 않아요. **채널 40은 그대로 두세요.** 오늘은 올바른 볼륨 변경이 아니라 입력 검사를 확인해요.


## 수정본으로 바꾸려면

GitHub 원본이 바뀌어도 이미 만든 Drive 사본은 자동으로 바뀌지 않아요. 적어 둔 답과 코드를 먼저 저장하세요. [최신 원본 열기](https://colab.research.google.com/github/GoBeromsu/jnu-llmops-precourse-day3/blob/main/notebooks/day3_mcp.ipynb)에서 새 Drive 사본을 만든 뒤, 내 답과 수정한 조건만 옮기고 첫 코드 칸부터 실행하세요. 기존 사본을 지우거나 덮어쓰지 않아도 돼요.


In [ ]:
# 이 칸은 MCP·Node.js와 서버 파일을 준비해요. 내 컴퓨터가 아니라 Colab에 설치돼요.
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import sysconfig

packages = {"mcp": "1.26.0", "nodejs-wheel": "22.14.0"}
missing = []
for name, version in packages.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{name}=={version}")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

# 방금 설치한 실행 파일을 먼저 찾게 해요.
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
assert node_version == "v22.14.0", node_version
npm = shutil.which("npm")
assert npm, "실행 파일을 찾지 못했어요. 설치 오류를 강사에게 보여 주세요."

# 서버를 지금 여기에 내려받아요. 서버를 실행하지는 않아요.
server_home = os.path.join(os.getcwd(), "mcp-server")
server_entry = os.path.join(
    server_home, "node_modules", "x-m32-mcp-server", "dist", "index.js"
)
if not os.path.exists(server_entry):
    subprocess.run(
        [npm, "install", "--prefix", server_home, "--no-audit", "--no-fund",
         "--loglevel=error", "x-m32-mcp-server@3.3.0"],
        check=True,
    )
assert os.path.exists(server_entry), server_entry
with open(os.path.join(server_home, "node_modules", "x-m32-mcp-server", "package.json"), encoding="utf-8") as f:
    installed_server = json.load(f)["version"]
if installed_server != "3.3.0":
    raise RuntimeError(f"서버 버전이 달라요: {installed_server}. 강사에게 보여 주세요.")
print("준비됐어요. 다음 코드 칸을 실행하세요.")

## 도구 24개를 받아요

다음 칸을 실행하면 도구 수와 `channel_set_volume`의 명세가 나와요. `name`, `description`, `inputSchema`를 찾아보세요.

이 칸은 첫 칸에서 내려받은 서버 파일을 실행해요. 60초가 지나도 답이 없으면 스스로 오류로 멈춰요. 그 메시지를 그대로 복사해 강사에게 보여 주세요. 첫 칸을 건너뛰었다면 먼저 첫 칸부터 실행하세요. 결과가 없는데 성공한 것으로 적지 않아요.

In [ ]:
# import는 다른 파일이나 패키지에 있는 이름을 여기서 쓰겠다는 뜻이에요.
import asyncio
import sys
import tempfile
from contextlib import asynccontextmanager
import json
import os
from datetime import timedelta
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.shared.exceptions import McpError

# 첫 칸에서 내려받은 파일을 Colab 안에서 실행해요. 장비 주소는 넘기지 않아요.
TIMEOUT = 60  # 답이 없으면 60초에 멈춰요. 안과 밖의 기다리는 시간을 같게 맞춰요.
server = StdioServerParameters(
    command=shutil.which("node"),
    args=[server_entry],
    env={"PATH": os.environ["PATH"], "HOME": os.environ["HOME"]},
)

# Colab의 화면 출력 스트림에는 운영체제 파일 번호가 없을 수 있어요.
# 서버 stderr는 실제 파일로 받고, 종료 후 내용을 화면에 그대로 보여 줘요.
@asynccontextmanager
async def notebook_stdio():
    with tempfile.TemporaryFile(mode="w+", encoding="utf-8") as errlog:
        try:
            async with stdio_client(server, errlog=errlog) as streams:
                yield streams
        finally:
            errlog.seek(0)
            diagnostics = errlog.read()
            if diagnostics:
                print(diagnostics, end="", file=sys.stderr)

async def read_tools():
    async with notebook_stdio() as (reader, writer):
        async with ClientSession(reader, writer, read_timeout_seconds=timedelta(seconds=TIMEOUT)) as session:
            await session.initialize()  # 먼저 서로 사용할 규격을 확인해요.
            response = await session.list_tools()  # tools/list 요청이에요.
            return response.tools

# await는 서버에서 결과가 올 때까지 기다려요.
tools = await asyncio.wait_for(read_tools(), timeout=TIMEOUT)
assert len(tools) == 24, f"예상한 도구는 24개인데 {len(tools)}개가 왔어요. 강사에게 알려 주세요."
print(f"도구 {len(tools)}개를 받았어요.")
volume = next(tool for tool in tools if tool.name == "channel_set_volume")
print(json.dumps({"name": volume.name, "description": volume.description,
                  "inputSchema": volume.inputSchema}, ensure_ascii=False, indent=2))

## 채널 40을 요청하면 어떻게 될까요?

명세에서 채널의 최댓값을 먼저 확인하세요. 다음 칸을 실행하면 실제 서버의 거부 메시지와 `확인했어요: 채널 40은 입력 검사에서 거부됐어요.` 두 줄이 나와요. 이 칸도 60초 동안 답이 없으면 오류로 멈춰요.

채널 번호를 정상 범위로 바꾸지 않아요. 이 실습에서는 믹서에 연결하지 않고, 허용 범위를 벗어난 입력만 보내요.

In [ ]:
async def check_invalid_channel():
    async with notebook_stdio() as (reader, writer):
        async with ClientSession(reader, writer, read_timeout_seconds=timedelta(seconds=TIMEOUT)) as session:
            await session.initialize()
            response = await session.list_tools()
            tool = next(t for t in response.tools if t.name == "channel_set_volume")
            # 규격이 바뀌면 요청하지 않고 멈춰요.
            assert tool.inputSchema["properties"]["channel"]["maximum"] == 32
            try:
                result = await session.call_tool(
                    "channel_set_volume", {"channel": 40, "value": -20, "unit": "db"}
                )
            except McpError as error:
                # MCP 프로토콜 오류로 돌아오는 경우예요.
                assert error.error.code == -32602, str(error)
                message = str(error)
            else:
                # 도구 실행 결과에 오류가 표시되는 경우예요.
                assert result.isError, "거부되지 않았어요. 강사에게 알려 주세요."
                message = "\n".join(c.text for c in result.content if c.type == "text")
            assert "32" in message and "channel" in message, message
            return message

message = await asyncio.wait_for(check_invalid_channel(), timeout=TIMEOUT)
print(message)
print("확인했어요: 채널 40은 입력 검사에서 거부됐어요.")

## 확인한 내용을 적어요

- 받은 도구 수:
- 볼륨 조절 Tool이 허용하는 채널 범위:
- 채널 40을 거부한 이유:

세 항목을 적으면 끝이에요. 오늘 확인한 것은 두 가지예요. MCP 서버가 **도구 목록과 명세를 알려 준다**는 것, 그리고 **범위를 벗어난 입력을 거부한다**는 것이에요.

오늘은 실제 믹서에 연결하지 않았고 올바른 요청을 보낸 적도 없어요. 그래서 이 결과로는 장비가 실제로 움직이는지까지는 확인할 수 없어요.

답을 적은 내용은 내 Drive 사본에 그대로 남아요. 하지만 설치한 프로그램과 내려받은 서버 파일은 런타임을 초기화하거나 오래 자리를 비우면 사라져요. 다음에 다시 할 때는 첫 번째 칸부터 순서대로 실행하세요.

출처: [XM32-MCP](https://github.com/GoBeromsu/XM32-MCP), [MCP Tools](https://modelcontextprotocol.io/specification/2025-06-18/server/tools).